# Первая нейронная сеть: Fashion-MNIST

Научимся распознавать вид одежды по изображению.
Сравним линейную модель и две небольшие нейросети, посмотрим ход обучения
и разберём несколько неверных ответов.

Данные лежат в папке `data`, интернет для их загрузки на занятии не нужен.

## 1. Подготовка

Запустите импорты. Обратите внимание на настройки: число эпох одинаково
у обеих сетей. В течение одной эпохи сеть проходит все учебные примеры
небольшими порциями и корректирует веса.

Для запуска нужны пакеты из `requirements.txt`.

In [ ]:
from pathlib import Path
from time import perf_counter
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from threadpoolctl import threadpool_limits

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

RANDOM_STATE = ваш_номер_из_списка_группы
VALID_SIZE = 2000
EPOCHS = количество_эпох_для_обучения
BATCH_SIZE = размер_пакета_для_обучения
MODEL_THREADS = 2
DATA_PATH = Path('data/fashion_mnist_subset.npz')
EXPORT_DIR = Path('exports')
CLASS_NAMES = np.array(['Футболка / топ', 'Брюки', 'Пуловер', 'Платье', 'Пальто',
                        'Сандалии', 'Рубашка', 'Кроссовки', 'Сумка', 'Ботильоны'])
CLASS_SHORT = np.array(['Топ', 'Брюки', 'Пуловер', 'Платье', 'Пальто',
                        'Сандалии', 'Рубашка', 'Кроссовки', 'Сумка', 'Ботильоны'])
CLASSES = np.arange(len(CLASS_NAMES))
BLUE, GRAY = '#005DAA', '#687789'

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlepad': 12, 'figure.dpi': 110})
pd.options.display.float_format = '{:.4f}'.format
assert DATA_PATH.exists(), 'Сохраните папку data рядом с блокнотом и запустите Jupyter из папки комплекта.'
print('Данные найдены:', DATA_PATH)

## 2. Изображения и классы

Fashion-MNIST содержит чёрно-белые изображения 28 × 28 пикселей и метки
десяти видов одежды и обуви. Здесь метка обозначает категорию товара.

В комплект вошли **12 000 изображений из исходных 60 000 учебных**:
по 1 200 каждого класса. Ещё 10 000 изображений официального теста
оставлены для итоговой проверки. Сейчас работаем только с учебной частью.

Посмотрите форму массива, тип пикселей и число изображений каждого класса.
На картинках ниже показаны известные метки из набора данных.

In [ ]:
with np.load(DATA_PATH, allow_pickle=False) as data:
    images_pool = data['images_pool']
    y_pool = data['labels_pool']
    pool_source_indices = data['pool_source_indices']

print('Изображения:', images_pool.shape, 'Тип:', images_pool.dtype)
print('Яркость пикселей:', images_pool.min(), '…', images_pool.max())
assert images_pool.shape == (12000, 28, 28)
assert images_pool.dtype == np.uint8 and np.isfinite(images_pool).all()
assert np.array_equal(np.unique(y_pool), CLASSES)
counts = pd.DataFrame({'Класс': CLASSES, 'Название': CLASS_NAMES,
                       'Изображений': np.bincount(y_pool, minlength=10)})
display(counts)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for label, ax in enumerate(axes.flat):
    position = np.flatnonzero(y_pool == label)[0]
    ax.imshow(images_pool[position], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{label}: {CLASS_NAMES[label]}', fontsize=11)
    ax.axis('off')
fig.suptitle('Примеры Fashion-MNIST: известные классы', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Подготовка входа и разделение данных

Для полносвязной сети превратите каждое изображение в строку из 784 чисел.
Порядок пикселей должен оставаться одинаковым у всех изображений.
Переведите массив в `float32` и разделите яркость на 255. Получится шкала от 0 до 1.

Из 12 000 записей выделите 2 000 для валидации. Признаки, метки и исходные
номера строк делите одним вызовом `train_test_split`. Используйте `stratify=y_pool`,
чтобы сохранить доли классов. Тест пока не загружаем.

Валидация нужна для выбора настроек. Итоговый тест используется после этого выбора.

In [ ]:
X_pool = ...  # изображения в строки, float32, яркость от 0 до 1
X_train, X_valid, y_train, y_valid, train_ids, valid_ids = train_test_split(
    ..., ..., pool_source_indices,
    test_size=VALID_SIZE, random_state=RANDOM_STATE, stratify=y_pool,
)
assert X_train.shape == (10000, 784) and X_valid.shape == (2000, 784)
assert X_train.dtype == np.float32 and 0 <= X_train.min() <= X_train.max() <= 1
assert set(train_ids).isdisjoint(valid_ids)
print('Обучение:', X_train.shape, 'Валидация:', X_valid.shape)
print('Первые 12 значений первой записи:', X_train[0, :12])

**Коротко запишите:** откуда взялось число 784? Почему метка класса хранится
отдельно от пикселей? Изменилось ли содержание изображения после деления на 255?

## 4. Линейная модель для сравнения

Обучите `LogisticRegression` на `X_train` и `y_train`, получите ответы для
`X_valid`. Здесь это классификатор, несмотря на слово Regression в названии.

Посчитайте accuracy на валидации: долю правильных ответов. Мы будем сравнивать
нейросеть с этой моделью на одних данных. Заранее ожидать победу нейросети не нужно.

In [ ]:
linear = LogisticRegression(max_iter=500, tol=0.001, random_state=RANDOM_STATE)
started = perf_counter()
with threadpool_limits(limits=MODEL_THREADS):
    ...  # обучить linear
    y_valid_linear = ...  # ответы linear для валидации
linear_seconds = perf_counter() - started
print(f'Accuracy, валидация: {accuracy_score(y_valid, y_valid_linear):.2%}')
print(f'Обучение и прогноз заняли {linear_seconds:.1f} с на этой машине')

## 5. Сеть с одним скрытым слоем

На входе 784 значения пикселей. В скрытом слое 16 нейронов: каждый вычисляет
взвешенную сумму входов и применяет нелинейную функцию ReLU. На выходе сеть
выдаёт оценки десяти классов.

Функция `train_network` готова. Внутри `partial_fit` продолжает обучение
на одну эпоху. Обычный повторный вызов `fit` для этой задачи не используем:
нам нужно сохранить веса и состояние обучения между эпохами.

Допишите вызов функции для `small_network`. В каждой эпохе мы только измеряем
качество на валидации. Веса меняются по учебным примерам.

In [ ]:
def train_network(model, X_train, y_train, X_valid, y_valid):
    history_rows = []
    started = perf_counter()
    with threadpool_limits(limits=MODEL_THREADS):
        for epoch in range(1, EPOCHS + 1):
            model.partial_fit(X_train, y_train, classes=CLASSES)
            history_rows.append({
                'Эпоха': epoch,
                'Функция потерь, обучение': model.loss_,
                'Accuracy, обучение': accuracy_score(y_train, model.predict(X_train)),
                'Accuracy, валидация': accuracy_score(y_valid, model.predict(X_valid)),
            })
            if epoch % 5 == 0:
                print(f"Эпоха {epoch:2d}: loss={model.loss_:.3f}, "
                      f"accuracy на валидации={history_rows[-1]['Accuracy, валидация']:.2%}")
    print(f'Время обучения и измерений: {perf_counter() - started:.1f} с')
    return pd.DataFrame(history_rows)

In [ ]:
small_network = MLPClassifier(
    hidden_layer_sizes=(16,), activation='relu', solver='adam',
    batch_size=BATCH_SIZE, learning_rate_init=0.001, alpha=0.0001,
    shuffle=True, random_state=RANDOM_STATE,
)
history_small = ...  # train_network: модель и четыре учебных/валидационных массива
print('Матрицы весов:', [weights.shape for weights in small_network.coefs_])
print('Обучаемых параметров:', sum(w.size for w in small_network.coefs_)
      + sum(b.size for b in small_network.intercepts_))

## 6. Ход обучения

Постройте готовые графики. Слева показана функция потерь на обучении:
она учитывает, какие оценки сеть даёт классам, и штраф за большие веса.
Справа accuracy на учебных и валидационных примерах.

Падение loss само по себе не гарантирует улучшение на новых изображениях.
Посмотрите, движутся ли обе кривые accuracy одинаково. В основной работе
каждая сеть проходит ровно 25 эпох; здесь нет выбора лучшей эпохи по графику.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history_small['Эпоха'], history_small['Функция потерь, обучение'], color=BLUE)
axes[0].set(title='MLP с 16 нейронами: функция потерь',
            xlabel='Эпоха', ylabel='Loss на обучении')
axes[1].plot(history_small['Эпоха'], history_small['Accuracy, обучение'],
             color=GRAY, linestyle='--', label='Обучение')
axes[1].plot(history_small['Эпоха'], history_small['Accuracy, валидация'],
             color=BLUE, label='Валидация')
axes[1].set(title='MLP с 16 нейронами: правильные ответы',
            xlabel='Эпоха', ylabel='Accuracy', ylim=(0, 1))
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1].legend()
plt.tight_layout()
plt.show()

**По своим графикам:** как изменилась loss? Есть ли разрыв между обучением
и валидацией? Можно ли считать, что ещё сто эпох обязательно улучшат результат?

## 7. Больше нейронов

Обучите вторую сеть с 64 нейронами. Число эпох, шаг обучения, исходные
примеры и остальные настройки оставьте прежними. Здесь мы меняем только
ширину скрытого слоя.

Соберите таблицу для линейной модели и двух сетей. Accuracy показывает
долю правильных ответов. F1 macro усредняет F1 десяти классов с одинаковым
весом: посмотрите, согласуется ли она с accuracy.

In [ ]:
wide_network = MLPClassifier(
    hidden_layer_sizes=(64,), activation='relu', solver='adam',
    batch_size=BATCH_SIZE, learning_rate_init=0.001, alpha=0.0001,
    shuffle=True, random_state=RANDOM_STATE,
)
history_wide = ...  # продолжительность и данные такие же, как у первой сети

In [ ]:
models = {'Линейная модель': linear, 'MLP 16': small_network, 'MLP 64': wide_network}
validation_rows = []
with threadpool_limits(limits=MODEL_THREADS):
    for name, model in models.items():
        valid_prediction = model.predict(X_valid)
        validation_rows.append({
            'Модель': name,
            'Accuracy, обучение': ...,
            'Accuracy, валидация': ...,
            'F1 macro, валидация': ...,
        })
validation_results = pd.DataFrame(validation_rows)
display(validation_results.round(4))

In [ ]:
comparison = validation_results.set_index('Модель')[['Accuracy, обучение', 'Accuracy, валидация']]
comparison.columns = ['Обучение', 'Валидация']
ax = comparison.plot.barh(color=[GRAY, BLUE], figsize=(10, 4.5), width=0.7)
ax.set(title='Сравнение на одних данных', xlabel='Accuracy', ylabel='', xlim=(0, 1))
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

## 8. Выбор размера сети

Из двух сетей выберите вариант с большей accuracy на валидации. При равенстве
оставьте меньшую сеть. Линейную модель сохраним как отдельный ориентир.

Запишите имя выбранной сети в `best_network_name`, затем получите её из
словаря `models`. Продолжаем работать с уже обученной моделью. Валидационные
записи в её обучение не добавляем.

In [ ]:
network_results = validation_results[validation_results['Модель'].isin(['MLP 16', 'MLP 64'])]
best_network_name = ...  # имя сети по правилу из задания
selected_network = models[best_network_name]
print('Выбор по валидации:', best_network_name)

## 9. Итоговая проверка

Теперь загрузите официальный тест. Для него нужно точно такое же
преобразование пикселей, как на шаге 3. Получите ответы выбранной сети,
посчитайте accuracy и F1 macro. Сравнение с линейной моделью готово.

После просмотра теста основной эксперимент закончен. Если снова менять
сеть ради лучшего тестового результата, потребуется новая независимая проверка.

В матрице ошибок строка обозначает известный класс, столбец - ответ сети.
Числа на диагонали соответствуют правильным ответам.

In [ ]:
with np.load(DATA_PATH, allow_pickle=False) as data:
    images_test = data['images_test']
    y_test = data['labels_test']
    test_ids = data['test_source_indices']
X_test = ...  # та же форма, тип и шкала, что у учебных данных
with threadpool_limits(limits=MODEL_THREADS):
    y_pred_test = ...  # ответы выбранной сети
    y_pred_linear_test = linear.predict(X_test)
    test_probabilities = selected_network.predict_proba(X_test)
test_results = pd.DataFrame([
    {'Модель': 'Линейная модель', 'Accuracy, тест': accuracy_score(y_test, y_pred_linear_test),
     'F1 macro, тест': f1_score(y_test, y_pred_linear_test, average='macro', zero_division=0)},
    {'Модель': best_network_name, 'Accuracy, тест': ...,
     'F1 macro, тест': ...},
])
assert len(y_pred_test) == len(y_test) == 10000
assert np.allclose(test_probabilities.sum(axis=1), 1, atol=1e-5)
display(test_results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
matrix_display = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test, labels=CLASSES, display_labels=CLASS_SHORT,
    cmap='Blues', colorbar=False, values_format='d',
    xticks_rotation=45, ax=ax, text_kw={'fontsize': 9},
)
ax.set(title=f'Официальный тест: {best_network_name}',
       xlabel='Ответ сети', ylabel='Известный класс')
plt.tight_layout()
plt.show()
assert matrix_display.confusion_matrix.sum() == len(y_test)

**Объясните по матрице:** какие классы сеть путает? Назовите одну клетку
вне диагонали словами. Почему общей accuracy недостаточно для описания всех ошибок?

## 10. Изображения с ошибками

Найдите позиции, где `y_pred_test` отличается от `y_test`. Покажите первые
шесть ошибок в порядке тестового набора. Изображения, правильные классы и
ответы сети берите по одним позициям.

Под каждым изображением приведена наибольшая оценка класса от `predict_proba`.
Гарантирует ли это правильный ответ?

In [ ]:
mistake_positions = ...  # номера позиций с неверным ответом
print('Неверных ответов:', len(mistake_positions), 'из', len(y_test))
error_positions = mistake_positions[:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, position in zip(axes.flat, error_positions):
    predicted = y_pred_test[position]
    score = test_probabilities[position].max()
    ax.imshow(images_test[position], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'Тест № {test_ids[position]}\n'
                 f'Известно: {CLASS_SHORT[y_test[position]]}\n'
                 f'Ответ: {CLASS_SHORT[predicted]}, оценка {score:.0%}', fontsize=10)
    ax.axis('off')
for ax in axes.flat[len(error_positions):]:
    ax.axis('off')
fig.suptitle('Первые шесть ошибок в порядке тестового набора', fontsize=14)
plt.tight_layout()
plt.show()

Выберите одну ошибку. Какие детали изображения могли затруднить различение?
Запишите предположение и отделите его от того, что вы действительно видите.

## 11. Один прогноз и экспорт

Подготовьте первое тестовое изображение как массив формы `(1, 784)`
в той же шкале от 0 до 1. Получите оценки классов и покажите три наибольшие.
Номер столбца вероятностей связывайте с `selected_network.classes_`.

Это демонстрация применения уже проверенной модели к одной записи из того же
теста. Она не является дополнительной независимой проверкой.

Запустите готовый экспорт. В папке `exports` появятся четыре CSV.

In [ ]:
example_position = 0
new_image = images_test[example_position]
new_X = ...  # одна строка, 784 признака, float32, шкала от 0 до 1
with threadpool_limits(limits=MODEL_THREADS):
    example_probabilities = ...  # оценки десяти классов для одной записи
top_positions = np.argsort(example_probabilities)[::-1][:3]
top_classes = selected_network.classes_[top_positions]
display(pd.DataFrame({'Класс': CLASS_NAMES[top_classes],
                      'Оценка': example_probabilities[top_positions]}))
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(new_image, cmap='gray', vmin=0, vmax=255)
ax.set_title(f'Тест № {test_ids[example_position]}\n'
             f'Известно: {CLASS_NAMES[y_test[example_position]]}', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
EXPORT_DIR.mkdir(exist_ok=True)
validation_results.to_csv(EXPORT_DIR / 'validation_comparison.csv', index=False, encoding='utf-8-sig')
test_results.to_csv(EXPORT_DIR / 'test_metrics.csv', index=False, encoding='utf-8-sig')
test_predictions = pd.DataFrame({
    'test_index': test_ids,
    'known_class': y_test, 'known_name': CLASS_NAMES[y_test],
    'predicted_class': y_pred_test, 'predicted_name': CLASS_NAMES[y_pred_test],
    'max_probability': test_probabilities.max(axis=1),
    'correct': y_pred_test == y_test,
})
test_predictions.to_csv(EXPORT_DIR / 'test_predictions.csv', index=False, encoding='utf-8-sig')
training_history = pd.concat([
    history_small.assign(Модель='MLP 16'), history_wide.assign(Модель='MLP 64')], ignore_index=True)
training_history.to_csv(EXPORT_DIR / 'training_history.csv', index=False, encoding='utf-8-sig')
print('Сохранено в папку:', EXPORT_DIR.resolve())
print('validation_comparison.csv, test_metrics.csv, test_predictions.csv, training_history.csv')

## 12. Вывод

Напишите выводы по своим результатам. Укажите, какую сеть выбрали
и по какой выборке. Сравните её с линейной моделью, опишите ход обучения
и одну конкретную ошибку. Назовите ограничение проведённой проверки.

**Мой вывод:**

_Здесь запишите наблюдения и значения из своего запуска._

## Если осталось время

В отдельном эксперименте исследуйте другую ширину слоя или число эпох,
используя только обучение и валидацию. Сохраните основной результат отдельно.
Повторный выбор настроек по уже просмотренному тесту не даёт новой независимой оценки.

Сеть в этой работе полносвязная. Она получает строку пикселей и не использует
свёртки. Качество на Fashion-MNIST не переносится автоматически на фотографии
с телефона: фон, положение предмета, цвет и освещение будут другими.

Источники: [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist),
[MLPClassifier и partial_fit](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html),
[правила проверки моделей](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).
Происхождение данных и контрольная сумма находятся в `data/source.json`.